### Introduction to NVIDIA cuOpt and MIP

NVIDIA cuOpt is a high-performance, GPU-accelerated combinatorial optimization engine. Unlike traditional solvers strictly focused on CPUs (such as Gurobi or CPLEX in their default modes), which expend significant computational time trying to mathematically prove the exact optimal bound through complex Branch and Bound routines, cuOpt's MIP (Mixed-Integer Programming) module focuses its horsepower on hardware-accelerated primal heuristics. It utilizes a powerful combination of evolutionary algorithms, parallel domain propagation, and the Primal-Dual Hybrid Gradient (PDLP) method. This architecture allows it to find extremely high-quality feasible solutions in fractions of a second, even for industrial-scale problems with millions of variables and constraints.

#### Feed Formulation (Mixed-Integer Linear Programming - MILP)

In Operations Research, this is a classic Diet or Blending Problem. However, it introduces a layer of real-world complexity by using Mixed Variables (MILP): one continuous variable and one discrete integer variable.
In livestock and animal production, feed represents the highest running cost. Nutritionists must design a feed blend that meets strict minimum nutritional thresholds (like protein, fiber, and energy) while tracking how raw materials are physically purchased in the supply chain. While some bulk commodities can be weighed precisely down to the gram, specialized micro-ingredients or concentrates are often strictly sold in fixed, whole packages.
Here is how your mathematical model handles this balancing act on the GPU:

1. The Core Objective

The goal is to formulate a specific batch of livestock feed at the absolute lowest cost possible. The optimization direction is set to Minimization (MINIMIZE) to shave off every possible cent from the production cost.

2. The Resource Profiles & Mixed Variables

The recipe draws from two distinct components with different physical purchasing rules:

• Corn Bulk Feed (Kg_Milho): A continuous commodity (vtype=CONTINUOUS). It costs R$ 2.50 per kg, and since it comes from a bulk silo, the factory can dispense any fractional amount needed (e.g., 412.35 kg).

• Concentrated Nutrient Additive (Sacos_Aditivo): A discrete integer ingredient (vtype=INTEGER). Each bag weighs exactly 5 kg and costs R$ 45.00. Because it is a commercial pre-packaged item, the factory cannot open a bag to use a fraction; it must buy and drop whole bags into the mixer.


3. The Boundary Constraints (The Quality Rules)

The recipe must strictly satisfy two nutritional and structural laws:

1. Total Batch Weight: The final mixture must weigh exactly 500 kg (== 500). Every bag of additive tossed in contributes 5 kg, and the rest is filled with bulk corn.
   
3. Minimum Crude Protein Target: The livestock requires at least 65 kg of pure protein within this 500 kg batch to hit growth targets. Bulk corn contributes 10% protein per kg, while the high-density additive yields 40% protein per bag (which mathematically equals 2 kg of pure protein inside each 5 kg bag).


##### Libraries

In [1]:
from cuopt.linear_programming.problem import Problem, INTEGER, CONTINUOUS, MINIMIZE
from cuopt.linear_programming.solver_settings import SolverSettings

##### Instantiating the Problem

In [11]:
# Initialize the Mixed-Integer Feed Blending problem
feed_problem = Problem("Feed_Formulation_MILP")

##### Declaring Variables

In [12]:
# 1. Define mixed variables with explicit bounds to ensure GPU compatibility
# Bulk corn can be fractional (CONTINUOUS), Additive must be whole bags (INTEGER)
kg_corn = feed_problem.addVariable(vtype=CONTINUOUS, lb=0.0, ub=1000.0, name="Kg_Corn")
bags_additive = feed_problem.addVariable(vtype=INTEGER, lb=0, ub=1000, name="Bags_Additive")


##### Objective Function

In [13]:
# 2. Set Objective Function: Minimize the combined blending cost
# Price: R$ 2.50 per kg of corn | R$ 45.00 per whole bag of additive
feed_problem.setObjective(2.5 * kg_corn + 45.0 * bags_additive, sense=MINIMIZE)


##### Constraints

In [14]:
# 3. Add Operational and Nutritional Constraints
# Rule 1: Total structural batch weight must equal exactly 500 kg
feed_problem.addConstraint(kg_corn + 5 * bags_additive == 500, name="Total_Batch_Weight")

# Rule 2: Nutritional requirement of at least 65 kg of protein
# Corn is 10% protein (0.10/kg) | Additive yields 2 kg of protein per bag
feed_problem.addConstraint(0.10 * kg_corn + 2.0 * bags_additive >= 65, name="Protein_Requirement")



##### Solving

In [15]:
# 4. Execute the optimization on the NVIDIA GPU
settings = SolverSettings()
feed_problem.solve(settings)

cuOpt version: 26.8.0, git hash: 400863c1, host arch: x86_64, device archs: 70-real,75-real,80-real,86-real,90a-real,100f-real,120a-real,120
CPU: Intel(R) Xeon(R) Platinum 8358 CPU @ 2.60GHz, threads (physical/logical): 15/30, RAM (available/total): 228.79 / 235.93 GiB
CUDA 12.9, device: NVIDIA A10 (ID 0), VRAM: 22.06 GiB
CUDA device UUID: f14fa984-7330-7e34-247d-b8f6d381c7fe
Solving a problem with 2 constraints, 2 variables (1 integers), and 4 nonzeros
Problem scaling:
Objective coefficents range:          [2e+00, 4e+01]
Constraint matrix coefficients range: [1e-01, 5e+00]
Constraint rhs / bounds range:        [0e+00, 5e+02]
Variable bounds range:                [0e+00, 1e+03]

MIP row scaling completed
New solution from early primal heuristics (CPUFJ). Objective +4.500000e+03. Time 0.00
New solution from early primal heuristics (CPUFJ). Objective +2.322500e+03. Time 0.00
New solution from early primal heuristics (CPUFJ). Objective +1.770000e+03. Time 0.01
New solution from early prim

In [16]:
# 5. Extract and print results safely using the 26.8.0 API
status_name = str(feed_problem.Status.name)
print("--- SOLVER EXECUTION STATUS ---")
print(f"Solution Status: {status_name}")

if "Optimal" in status_name or "Feasible" in status_name:
    print("\n--- OPTIMAL FEED RECIPE FOUND ---")
    print(f"Bulk Corn required: {kg_corn.getValue():.2f} Kg")
    # Using ':.0f' to elegantly handle floating-point precision on the integer variable
    print(f"Packaged Additive required: {bags_additive.getValue():.0f} bags")
    
    formatted_cost = "{:,.2f}".format(feed_problem.ObjValue)
    print(f"Minimum Production Cost: R$ {formatted_cost}")
    
    # Internal quality assurance check
    total_weight = kg_corn.getValue() + (5 * bags_additive.getValue())
    total_protein = (0.10 * kg_corn.getValue()) + (2.0 * bags_additive.getValue())
    print("\n--- QUALITY ASSURANCE ---")
    print(f"Calculated Batch Weight: {total_weight:.1f} kg (Target: 500.0)")
    print(f"Calculated Protein Content: {total_protein:.2f} kg (Min Target: 65.0)")
else:
    print("\nInfeasible: The solver could not find a mix that hits nutritional targets under these weight limits.")

--- SOLVER EXECUTION STATUS ---
Solution Status: Optimal

--- OPTIMAL FEED RECIPE FOUND ---
Bulk Corn required: 450.00 Kg
Packaged Additive required: 10 bags
Minimum Production Cost: R$ 1,575.00

--- QUALITY ASSURANCE ---
Calculated Batch Weight: 500.0 kg (Target: 500.0)
Calculated Protein Content: 65.00 kg (Min Target: 65.0)
